# Sample End-to-End ETL Execution

This notebook exercises the FULL metadata-driven pipeline using the **EMPLOYEE_MASTER** sample group defined in `setup/metadata_setup_sample.sql`.

Pipeline flow:
```
  EMPLOYEE_MASTER_L0 (L0: bronze CSV from GitHub)
        ↓
  EMPLOYEE_MASTER_L1 (L1: silver.employee_master_clean, MERGE)
        ↓
  EMPLOYEE_MASTER_L2 (L2: gold.dim_employee SCD2 + gold.fact_hire_stats)
```

All 3 framework tables are used, and every column in each table is honored. After each run we inspect the audit trail.

In [ ]:
import sys, os, json, traceback
from datetime import datetime

dbutils.widgets.dropdown("RUN", "ALL", ["ALL", "L0_ONLY", "L1_ONLY", "L2_ONLY", "RESET"], "Run mode")
RUN_MODE = dbutils.widgets.get("RUN").strip().upper()

try:
    nb_path = json.loads(
        dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson()
    ).get("extraContext", {}).get("notebook_path", "")
except Exception:
    nb_path = ""
FRAMEWORK_BASE = os.path.dirname(nb_path) if nb_path else "/Workspace/Repos"
if FRAMEWORK_BASE and FRAMEWORK_BASE not in sys.path:
    sys.path.insert(0, FRAMEWORK_BASE)

CATALOG        = "demo_catalog"
CONTROL_SCHEMA = "admin"
ENV            = "dev"

from framework.utils.structured_logger import StructuredLogger
from framework.utils.audit_manager    import AuditManager
from framework.utils.metadata_validator import MetadataValidator
from framework.utils.spark_utils      import SparkUtils
from framework.core.orchestrator_core import OrchestratorCore
from framework.layers.l0_processor    import L0Processor
from framework.layers.l1l2_processor  import L1L2Processor

logger = StructuredLogger(group_id="SAMPLE_RUN", layer="ALL",
                          catalog=CATALOG, control_schema=CONTROL_SCHEMA)
audit  = AuditManager(spark, CATALOG, CONTROL_SCHEMA)
orch   = OrchestratorCore(spark, dbutils, CATALOG, CONTROL_SCHEMA, logger, audit)

print(f"FRAMEWORK_BASE = {FRAMEWORK_BASE}")
print(f"RUN_MODE       = {RUN_MODE}")
print(f"Catalog        = {CATALOG}")

In [ ]:
if RUN_MODE == "RESET":
    print("=== Soft RESET: drop bronze / silver / gold EMPLOYEE tables (NOT framework tables) ===")
    for stmt in [
        "DROP TABLE IF EXISTS demo_catalog.bronze.employee_master_data_messy_10000",
        "DROP TABLE IF EXISTS demo_catalog.silver.employee_master_clean",
        "DROP TABLE IF EXISTS demo_catalog.gold.dim_employee",
        "DROP TABLE IF EXISTS demo_catalog.gold.fact_hire_stats",
    ]:
        print(f"  {stmt}")
        spark.sql(stmt)
    print("Done.")

In [ ]:
if RUN_MODE != "RESET":
    print("=== 1. Validate metadata for the 3 EMPLOYEE_MASTER groups ===")
    for gid in ("EMPLOYEE_MASTER_L0", "EMPLOYEE_MASTER_L1", "EMPLOYEE_MASTER_L2"):
        print(f"\n--- {gid} ---")
        hdr = orch.fetch_control_header(gid, validate=True)
        print(f"  control_header  -> ok (layer={hdr['ETL_LAYER']}, trigger={hdr['TRIGGER_TYPE']}, target_catalog={hdr.get('target_catalog')})")
        if gid.endswith("_L0"):
            recs = orch.fetch_l0_details(gid, validate=True)
            for r in recs:
                print(f"  l0_detail row   -> {r['SOURCE_OBJ_SCHEMA']}.{r['SOURCE_OBJ_NAME']}  LOAD_TYPE={r['LOAD_TYPE']}  FORMAT={r['INPUT_FILE_FORMAT']}")
        else:
            layer = gid[-2:]
            recs = orch.fetch_pb_details(gid, layer=layer, validate=True)
            for r in recs:
                pk = r.get('TARGET_PK') or r.get('SOURCE_PK') or '-'
                print(f"  pb_detail row   -> p{r.get('PRIORITY') or 999} {r['TARGET_OBJ_SCHEMA']}.{r['TARGET_OBJ_NAME']}  TYPE={r['TARGET_OBJ_TYPE']} LOAD={r['LOAD_TYPE']} PK={pk}")

In [ ]:
l0_results = []
if RUN_MODE in ("ALL", "L0_ONLY"):
    print("\n=== 2. L0 Bronze Ingestion (EMPLOYEE_MASTER_L0) ===")
    l0p = L0Processor(spark, dbutils,
                      catalog=CATALOG, control_schema=CONTROL_SCHEMA,
                      logger=logger, audit_manager=audit, environment=ENV)
    try:
        l0_results = l0p.process_group("EMPLOYEE_MASTER_L0")
    except Exception as e:
        print(f"L0 FAILED but continuing for visibility: {e}")

    if l0_results:
        print("\nL0 result row count preview (top 5 bronze rows):")
        for r in l0_results:
            if r["status"] == "SUCCESS":
                tbl = f"{CATALOG}.{r['target_schema']}.{r['target_table']}"
                try:
                    spark.sql(f"SELECT * FROM {tbl} LIMIT 5").display()
                except Exception as disp_err:
                    print(f"  (display failed: {disp_err})")

In [ ]:
l1_results = []
if RUN_MODE in ("ALL", "L1_ONLY"):
    print("\n=== 3. L1 Silver Clean (EMPLOYEE_MASTER_L1) ===")
    l1p = L1L2Processor("L1", spark=spark, dbutils=dbutils,
                        catalog=CATALOG, control_schema=CONTROL_SCHEMA,
                        logger=logger, audit_manager=audit, environment=ENV)
    try:
        l1_results = l1p.process_group("EMPLOYEE_MASTER_L1", run_by_priority=True)
    except Exception as e:
        print(f"L1 FAILED but continuing for visibility: {e}")

    if l1_results:
        for r in l1_results:
            if r["status"] == "SUCCESS":
                tbl = f"{CATALOG}.{r['target_schema']}.{r['target_table']}"
                print(f"\nL1 {tbl}: count = {r['rows_processed']:,}, top 5 rows:")
                try:
                    spark.sql(f"SELECT * FROM {tbl} LIMIT 5").display()
                    spark.sql(f"SELECT employment_status, COUNT(*) AS n FROM {tbl} GROUP BY employment_status").display()
                except Exception as disp_err:
                    print(f"  (display failed: {disp_err})")

In [ ]:
l2_results = []
if RUN_MODE in ("ALL", "L2_ONLY"):
    print("\n=== 4. L2 Gold Marts (EMPLOYEE_MASTER_L2) priority-based ===")
    l2p = L1L2Processor("L2", spark=spark, dbutils=dbutils,
                        catalog=CATALOG, control_schema=CONTROL_SCHEMA,
                        logger=logger, audit_manager=audit, environment=ENV)
    try:
        l2_results = l2p.process_group("EMPLOYEE_MASTER_L2", run_by_priority=True)
    except Exception as e:
        print(f"L2 FAILED but continuing for visibility: {e}")

    if l2_results:
        for r in l2_results:
            if r["status"] == "SUCCESS":
                tbl = f"{CATALOG}.{r['target_schema']}.{r['target_table']}"
                print(f"\nL2 {tbl}: count = {r['rows_processed']:,}, top 5 rows:")
                try:
                    spark.sql(f"SELECT * FROM {tbl} ORDER BY total_employees DESC LIMIT 10").display()
                except Exception as disp_err:
                    print(f"  (display failed: {disp_err})")

In [ ]:
if RUN_MODE != "RESET":
    print("\n=== 5. Audit Trail  (demo_catalog.admin.audit_log) ===")
    groups_filter = "'EMPLOYEE_MASTER_L0','EMPLOYEE_MASTER_L1','EMPLOYEE_MASTER_L2'"
    audit_df = spark.sql(f"""
        SELECT
            ETL_LAYER,
            TARGET_TABLE,
            STATUS,
            ROWS_PROCESSED,
            DURATION_SECONDS,
            START_TIME,
            END_TIME,
            LEFT(MESSAGE, 160) AS MESSAGE_PREVIEW,
            ENVIRONMENT,
            LOB
        FROM demo_catalog.admin.audit_log
        WHERE DATA_FLOW_GROUP_ID IN ({groups_filter})
        ORDER BY LOAD_TS DESC
        LIMIT 50
    """)
    display(audit_df)

    print("\n=== 6. Final table counts ===")
    for tbl in [
        "bronze.employee_master_data_messy_10000",
        "silver.employee_master_clean",
        "gold.dim_employee",
        "gold.fact_hire_stats",
    ]:
        try:
            c = spark.sql(f"SELECT COUNT(*) AS n FROM {CATALOG}.{tbl}").collect()[0][0]
            print(f"  {tbl:<55} = {c:,}")
        except Exception as ex:
            print(f"  {tbl:<55} = (table not found or failed: {ex})")